# ML-Based Anomaly Detection for Edge HTTP Traffic

**CloudWalk Case Study — Quick Start**

This notebook runs the entire pipeline end-to-end: data loading → label joining → feature engineering → model training → evaluation → ONNX export → monitoring.

Click **Runtime → Run all** to execute everything.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/felipetp-ctrl/cloudwalk-case-study/blob/main/notebooks/colab_quickstart.ipynb)

## 1. Setup

Clone the repository and install dependencies.

In [ ]:
import os

REPO_URL = "https://github.com/felipetp-ctrl/cloudwalk-case-study.git"
REPO_DIR = "cloudwalk-case-study"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
!pip install -q pandas numpy scikit-learn xgboost lightgbm optuna matplotlib seaborn onnxruntime skl2onnx onnxmltools scipy

In [ ]:
import sys
sys.path.insert(0, '.')

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.label_joining import join_labels
from src.features import compute_per_request_features, compute_session_features
from src.pipeline import build_training_dataset, compute_sample_weights
from src.model import (
    get_feature_columns, temporal_train_test_split,
    train_model, evaluate_model, evaluate_per_attack_type,
    get_feature_importance, make_time_series_cv_splits, tune_model,
)
from src.export import export_to_onnx, validate_onnx_export
from src.monitoring import (
    detect_data_drift, monitor_predictions,
    track_performance, generate_monitoring_report,
)

print('All imports successful.')

## 2. Data Loading & Label Joining

Join incident labels to individual HTTP requests using IP and CIDR matching with temporal bounds.

In [ ]:
requests = pd.read_csv('data/http_requests.csv', parse_dates=['timestamp'])
headers = pd.read_csv('data/request_headers.csv')
labels = pd.read_csv('data/incident_labels.csv', parse_dates=['active_from', 'active_until', 'labeled_at'])

print(f'Requests: {len(requests):,}')
print(f'Headers:  {len(headers):,}')
print(f'Labels:   {len(labels):,}')
print(f'\nAttack types: {labels["attack_class"].value_counts().to_dict()}')

In [ ]:
df = join_labels(requests, labels)

malicious = df['is_malicious'].sum()
benign = len(df) - malicious
print(f'Malicious: {malicious:,} ({malicious/len(df)*100:.2f}%)')
print(f'Benign:    {benign:,} ({benign/len(df)*100:.2f}%)')
print(f'\nAttack class breakdown:')
print(df[df['is_malicious'] == 1]['attack_class'].value_counts().to_string())

## 3. Feature Engineering

Extract 15 per-request features and 21 IP session features with causal windowing.

In [ ]:
df = compute_per_request_features(df, headers)
df = compute_session_features(df)
df['sample_weight'] = compute_sample_weights(df)

feature_cols = get_feature_columns(df)
print(f'Total features: {len(feature_cols)}')
print(f'\nFeature categories:')
print(f'  Per-request:  {sum(1 for c in feature_cols if not c.startswith("ip_"))}')
print(f'  IP session:   {sum(1 for c in feature_cols if c.startswith("ip_"))}')

## 4. Model Training & Evaluation

Temporal train/test split (train: days 6-9, test: days 10-12). Train LightGBM with Optuna hyperparameter tuning.

In [ ]:
train, test = temporal_train_test_split(df, '2025-01-10')
X_train, y_train = train[feature_cols], train['is_malicious'].astype(int)
X_test, y_test = test[feature_cols], test['is_malicious'].astype(int)

print(f'Train: {len(train):,} samples ({y_train.sum()} malicious)')
print(f'Test:  {len(test):,} samples ({y_test.sum()} malicious)')

In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

cv_splits = make_time_series_cv_splits(train)
best_params, study = tune_model(
    'lgbm', X_train, y_train, train['sample_weight'],
    cv_splits, n_trials=30,
)
best_params['verbose'] = -1
print(f'Best CV PR-AUC: {study.best_value:.4f}')
print(f'Best params: {best_params}')

In [ ]:
model = train_model('lgbm', best_params, X_train, y_train, train['sample_weight'])
metrics = evaluate_model(model, X_test, y_test)
per_attack = evaluate_per_attack_type(model, X_test, y_test, test['attack_class'])

print('=== Test Set Metrics ===')
for k, v in metrics.items():
    if k != 'y_prob':
        print(f'  {k:12s}: {v:.4f}')

print('\n=== Per-Attack Recall ===')
for cls, recall in per_attack.items():
    print(f'  {cls:25s}: {recall:.4f}')

## 5. Visualizations

In [ ]:
importance = get_feature_importance(model, 'lgbm', feature_cols)

fig, ax = plt.subplots(figsize=(10, 8))
importance.head(20).sort_values().plot.barh(ax=ax, color='#2196F3')
ax.set_title('Top 20 Features by LightGBM Gain')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import precision_recall_curve

precision, recall, _ = precision_recall_curve(y_test, metrics['y_prob'])

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(recall, precision, color='#4CAF50', lw=2, label=f'LightGBM (PR-AUC={metrics["pr_auc"]:.4f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curve')
ax.legend()
ax.set_xlim([0, 1.05])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.show()

## 6. ONNX Export & Validation

Export the model to ONNX format for edge deployment and validate numerical equivalence.

In [ ]:
onnx_path = export_to_onnx(model, feature_cols, 'model_colab.onnx')
result = validate_onnx_export(model, onnx_path, X_test)

print(f'ONNX model exported to: {onnx_path}')
print(f'Model size: {os.path.getsize(onnx_path) / 1024:.1f} KB')
print(f'Max abs error: {result["max_abs_error"]:.2e}')
print(f'Numerical match: {result["match"]}')

## 7. Model Monitoring

Demonstrate production monitoring capabilities: data drift detection, prediction stability, and performance tracking.

In [ ]:
drift_results = detect_data_drift(X_train, X_test)
n_drifted = drift_results['drifted'].sum()
print(f'Features with drift: {n_drifted}/{len(drift_results)}')
if n_drifted > 0:
    print('\nTop drifted features:')
    top = drift_results[drift_results['drifted']].sort_values('ks_statistic', ascending=False).head(5)
    for _, row in top.iterrows():
        print(f'  {row["feature"]:40s}  KS={row["ks_statistic"]:.4f}  p={row["p_value"]:.2e}')

In [ ]:
train_scores = model.predict_proba(X_train)[:, 1]
test_scores = model.predict_proba(X_test)[:, 1]

pred_report = monitor_predictions(train_scores, test_scores)
print('=== Prediction Stability ===')
print(f'  PSI:                {pred_report["psi"]:.4f}')
print(f'  Mean score (ref):   {pred_report["mean_score_reference"]:.4f}')
print(f'  Mean score (cur):   {pred_report["mean_score_current"]:.4f}')
print(f'  Alert rate (ref):   {pred_report["alert_rate_reference"]:.4f}')
print(f'  Alert rate (cur):   {pred_report["alert_rate_current"]:.4f}')

In [ ]:
baseline = {k: v for k, v in metrics.items() if k != 'y_prob'}
perf_windows = track_performance(y_test.values, metrics['y_prob'], window_size=500, baseline_metrics=baseline)

report = generate_monitoring_report(drift_results, pred_report, perf_windows)

import json
print('=== Monitoring Report ===')
print(json.dumps(report, indent=2))

## Summary

This notebook demonstrated the full ML pipeline:

1. **Label joining** — IP and CIDR matching with temporal bounds
2. **Feature engineering** — 36 features (15 per-request + 21 IP session) with causal windowing (no data leakage)
3. **Model training** — LightGBM with Optuna tuning and temporal cross-validation
4. **ONNX export** — Edge-ready model with numerical validation
5. **Monitoring** — Data drift, prediction stability, and performance tracking

For the full writeup, see [WRITEUP.md](https://github.com/felipetp-ctrl/cloudwalk-case-study/blob/main/WRITEUP.md).